# GoldenCheetah 데이터 탐색

## 분석 목적

GoldenCheetah 공개 데이터가 훈련 분석과 오늘의 훈련 추천 프로젝트에
사용할 수 있는지 확인한다.

## 현재까지 확인한 내용

- 전체 운동 기록: 731개
- 자전거 운동: 592개
- 파워 센서 포함: 469개
- 심박 센서 포함: 465개
- 파워와 심박 모두 포함: 373개
- 라이딩마다 센서와 요약 지표 구성이 다름

In [94]:
import sys
from pathlib import Path

import json

import pandas as pd

project_root = Path("..").resolve()

print("Python 경로:", sys.executable)
print("프로젝트 경로:", project_root)
print("pandas 버전:", pd.__version__)

json_path = (
    project_root
    / "data"
    / "raw"
    / "033874ce-e20d-44ba-9cc9-125030b6662f"
    / "{033874ce-e20d-44ba-9cc9-125030b6662f}.json"
)

Python 경로: /Users/hooni/Documents/ChatGPT/Cycling App/.venv/bin/python
프로젝트 경로: /Users/hooni/Documents/ChatGPT/Cycling App
pandas 버전: 3.0.5


## 1. JSON 데이터 불러오기

JSON 파일을 Python 자료형으로 불러온다.
전체 운동 중 `Bike` 기록만 선택하여 pandas 분석에 사용한다.

In [95]:
with json_path.open("r", encoding="utf-8") as file:
    cycling_data = json.load(file)

rides = cycling_data["RIDES"]

bike_rides = [ride for ride in rides if ride["sport"] == "Bike"]

print("전체 운동 수:", len(rides))
print("자전거 운동 수:", len(bike_rides))

전체 운동 수: 731
자전거 운동 수: 592


라이드 기록의 `data`는 15자리의 대문자 알파벳 문자열을 값으로 가지는데, 해당 운동에 어떤 센서 데이터가 포함되어 있는지 나타낸다.

| 문자 | 데이터 |
|---|---|
| `T` | 시간 |
| `D` | 거리 |
| `S` | 속도 |
| `P` | 파워 |
| `H` | 심박수 |
| `C` | 케이던스 |
| `N` | 토크 |
| `A` | 고도 |
| `G` | GPS |
| `L` | 경사도 |
| `W` | 풍속 |
| `E` | 온도 |
| `V` | 좌우 페달 데이터 |
| `O` | 근육 산소 관련 데이터 |
| `R` | Garmin 러닝 다이내믹스 |

In [96]:
first_ride = bike_rides[0]

print(first_ride.keys())
print("첫 라이드의 날짜", first_ride["date"])
print("첫 라이드 기록의 센서 목록:", first_ride["data"])

dict_keys(['date', 'data', 'sport', 'METRICS'])
첫 라이드의 날짜 2005/06/25 14:26:00 UTC
첫 라이드 기록의 센서 목록: TDS-H--A-L-----


첫 라이드의 경우 `TDS-H--A-L-----`로, 시간, 거리, 속도, 심박수, 고도, 경사도 데이터를 포함하고 있다.

## 2. 자전거 기록을 표로 변환

592개의 자전거 기록은 딕셔너리로 구성되어 있다.
pandas의 `DataFrame`을 이용하여 행과 열로 구성된 표로 변환한다.

In [97]:
bike_raw_df = pd.DataFrame(bike_rides)

print("자료형", type(bike_raw_df))
print("표 크기:", bike_raw_df.shape)
print("열 이름", bike_raw_df.columns)

bike_raw_df.head()

자료형 <class 'pandas.DataFrame'>
표 크기: (592, 5)
열 이름 Index(['date', 'data', 'sport', 'METRICS', 'XDATA'], dtype='str')


,date,data,sport,METRICS,XDATA
0,2005/06/25 14:26:00 UTC,TDS-H--A-L-----,Bike,"{'a_skiba_variability_index': 'nan', 'a_coggam...",NaN
1,2005/06/27 08:39:00 UTC,TDS-H--A-L-----,Bike,"{'a_skiba_variability_index': 'nan', 'a_coggam...",NaN
2,2005/06/28 17:44:00 UTC,TDS-HC-A-L-----,Bike,"{'a_skiba_variability_index': 'nan', 'a_coggam...",NaN
3,2005/07/06 18:02:00 UTC,TDS-H--A-L-----,Bike,"{'a_skiba_variability_index': 'nan', 'a_coggam...",NaN
4,2005/07/10 09:39:00 UTC,TDS-H--A-L-----,Bike,"{'a_skiba_variability_index': 'nan', 'a_coggam...",NaN


- 표는 592개의 행과 5개의 열로 구성되어 있다.
- 기본 열은 `date`, `data`, `sport`, `METRICS`, `XDATA`이다.
- `METRICS`는 아직 하나의 딕셔너리로 저장되어 있다.
- `XDATA`는 일부 라이딩에만 존재하므로 대부분 결측값으로 표시된다.

## 3. METRICS 펼치기

각 라이딩의 `METRICS`에는 운동 시간, 거리, 파워, 심박수 등
여러 요약 지표가 딕셔너리 형태로 저장되어 있다.

`METRICS`의 각 키를 DataFrame의 개별 열로 변환한다.

In [98]:
metrics_df = pd.json_normalize(bike_raw_df["METRICS"])

print("METRICS 표 크기", metrics_df.shape)
print("앞쪽 열 10개:")
print(metrics_df.columns[:10])

METRICS 표 크기 (592, 228)
앞쪽 열 10개:
Index(['a_skiba_variability_index', 'a_coggam_variability_index', 'ride_count',
       'workout_time', 'time_riding', 'total_distance', 'climb_rating',
       'athlete_weight', 'elevation_gain', 'elevation_loss'],
      dtype='str')


In [99]:
sample_columns = [
    "workout_time",
    'time_riding',
    "total_distance",
    'elevation_gain',
    "average_power",
    "average_hr",
    "coggan_tss",
    "coggan_if",
]

metrics_df[sample_columns].head()

,workout_time,time_riding,total_distance,elevation_gain,average_power,average_hr,coggan_tss,coggan_if
0,4800.00000,4780.00000,35.32750,367.00000,NaN,"[146.91667, 960.00000]",NaN,NaN
1,6476.00000,6325.00000,35.01400,467.50000,NaN,"[124.97267, 6476.00000]",NaN,NaN
2,2156.00000,2156.00000,17.65300,92.00000,NaN,"[157.04592, 2156.00000]",NaN,NaN
3,6360.00000,6320.00000,31.04950,512.00000,NaN,"[120.05660, 1272.00000]",NaN,NaN
4,4680.00000,4660.00000,32.77000,471.00000,NaN,"[146.42735, 936.00000]",NaN,NaN


- 592개 자전거 기록의 `METRICS`를 펼치자 228개의 지표가 나타났다.
- 라이딩마다 포함된 지표가 달라 전체 지표 수가 많아졌다.
- 이전 .py 파일에서 보았듯이, 평균 파워와 평균 심박은 일부 행에서 리스트 형태로 저장되어 있다.
- 파워 센서가 없는 초기 라이딩에서는 파워, TSS, IF가 결측값으로 표시된다.
- 228개 지표를 모두 사용하지 않고 분석 목적에 필요한 지표만 선택해야 한다.

## 4. 분석에 사용할 지표 선택

228개의 `METRICS` 지표들은 모두 서로 다른 센서 데이터가 아니라, 기본 센서값을
여러 계산 방식으로 요약한 지표도 포함하고 있다.

예를 들어 파워 관련 지표에는 다음과 같은 계산 계열이 존재한다.

- `coggan_`: Coggan 방식의 파워 강도와 훈련 부하 지표
- `skiba_`: Skiba 방식의 파워 강도와 훈련 부하 지표
- `a_`: 고도의 영향을 반영한 보정 지표

비슷한 의미의 지표를 모두 사용하면 정보가 중복될 수 있으므로,
첫 번째 분석에서는 해석하기 쉬운 기본 지표와 Coggan 계열을 우선 사용한다.

### 기본 정보

- `date`: 운동 날짜
- `data`: 기록된 센서 종류
- `sport`: 운동 종류

### 운동량

- `workout_time`: 전체 운동 시간
- `time_riding`: 실제 이동 시간
- `total_distance`: 총거리
- `elevation_gain`: 누적 상승고도
- `average_speed`: 평균 속도

### 파워

- `average_power`: 평균 파워
- `coggan_np`: 변동성을 고려한 대표 파워
- `max_power`: 최대 파워
- `cp_setting`: 해당 시점의 기준 파워

### 심박과 케이던스

- `average_hr`: 평균 심박수
- `max_heartrate`: 최대 심박수
- `average_cad`: 평균 케이던스
- `max_cadence`: 최대 케이던스

### 훈련 강도와 부하

- `coggan_if`: 기준 파워 대비 운동 강도
- `coggan_tss`: 운동 시간과 강도를 반영한 훈련 부하

`skiba_` 계열과 `a_` 계열은 원본에 보존하고, 이후 계산 방식과
고도 영향을 비교할 필요가 생기면 별도로 분석한다.

In [100]:
selected_metric_columns = [
    "workout_time",     # 전체 운동 시간
    "time_riding",      # 실제 이동 시간
    "total_distance",   # 총거리
    "elevation_gain",   # 획득 고도
    "average_speed",    # 평균 속도
    "average_power",    # 평균 파워
    "max_power",        # 최대 파워
    "coggan_np",        # NP
    "cp_setting",       # 해당 시점의 기준 파워
    "average_hr",       # 평균 심박수
    "max_heartrate",    # 최대 심박수
    "average_cad",      # 평균 케이던스
    "max_cadence",      # 최대 케이던스
    "coggan_if",        # IF
    "coggan_tss",       # TSS
]

selected_metrics_df = metrics_df[selected_metric_columns].copy()

print(selected_metrics_df.shape)
selected_metrics_df.head()

(592, 15)


,workout_time,time_riding,total_distance,elevation_gain,average_speed,average_power,max_power,coggan_np,cp_setting,average_hr,max_heartrate,average_cad,max_cadence,coggan_if,coggan_tss
0,4800.00000,4780.00000,35.32750,367.00000,26.60649,NaN,NaN,NaN,200.00000,"[146.91667, 960.00000]",184.00000,NaN,NaN,NaN,NaN
1,6476.00000,6325.00000,35.01400,467.50000,19.92892,NaN,NaN,NaN,200.00000,"[124.97267, 6476.00000]",182.00000,NaN,NaN,NaN,NaN
2,2156.00000,2156.00000,17.65300,92.00000,29.71052,NaN,NaN,NaN,200.00000,"[157.04592, 2156.00000]",168.00000,"[88.72929, 1919.00000]",98.00000,NaN,NaN
3,6360.00000,6320.00000,31.04950,512.00000,17.68642,NaN,NaN,NaN,200.00000,"[120.05660, 1272.00000]",182.00000,NaN,NaN,NaN,NaN
4,4680.00000,4660.00000,32.77000,471.00000,25.31588,NaN,NaN,NaN,200.00000,"[146.42735, 936.00000]",177.00000,NaN,NaN,NaN,NaN


In [101]:
metadata_df = bike_raw_df[
    ["data", "date", "sport"]
].copy()

analysis_df = pd.concat(
    [metadata_df, selected_metrics_df],
    axis=1
)

print(analysis_df.shape)
analysis_df.head()

(592, 18)


,data,date,sport,workout_time,time_riding,total_distance,elevation_gain,average_speed,average_power,max_power,coggan_np,cp_setting,average_hr,max_heartrate,average_cad,max_cadence,coggan_if,coggan_tss
0,TDS-H--A-L-----,2005/06/25 14:26:00 UTC,Bike,4800.00000,4780.00000,35.32750,367.00000,26.60649,NaN,NaN,NaN,200.00000,"[146.91667, 960.00000]",184.00000,NaN,NaN,NaN,NaN
1,TDS-H--A-L-----,2005/06/27 08:39:00 UTC,Bike,6476.00000,6325.00000,35.01400,467.50000,19.92892,NaN,NaN,NaN,200.00000,"[124.97267, 6476.00000]",182.00000,NaN,NaN,NaN,NaN
2,TDS-HC-A-L-----,2005/06/28 17:44:00 UTC,Bike,2156.00000,2156.00000,17.65300,92.00000,29.71052,NaN,NaN,NaN,200.00000,"[157.04592, 2156.00000]",168.00000,"[88.72929, 1919.00000]",98.00000,NaN,NaN
3,TDS-H--A-L-----,2005/07/06 18:02:00 UTC,Bike,6360.00000,6320.00000,31.04950,512.00000,17.68642,NaN,NaN,NaN,200.00000,"[120.05660, 1272.00000]",182.00000,NaN,NaN,NaN,NaN
4,TDS-H--A-L-----,2005/07/10 09:39:00 UTC,Bike,4680.00000,4660.00000,32.77000,471.00000,25.31588,NaN,NaN,NaN,200.00000,"[146.42735, 936.00000]",177.00000,NaN,NaN,NaN,NaN


In [102]:
print(analysis_df.dtypes)

data                 str
date                 str
sport                str
workout_time         str
time_riding          str
total_distance       str
elevation_gain       str
average_speed        str
average_power     object
max_power            str
coggan_np         object
cp_setting           str
average_hr        object
max_heartrate        str
average_cad       object
max_cadence          str
coggan_if         object
coggan_tss           str
dtype: object


In [103]:
for column in selected_metric_columns:
    values = analysis_df[column].dropna()
    first_value = values.iloc[0]

    print(
        column,
        "| 값:", first_value,
        "| 자료형:", type(first_value).__name__
    )

workout_time | 값: 4800.00000 | 자료형: str
time_riding | 값: 4780.00000 | 자료형: str
total_distance | 값: 35.32750 | 자료형: str
elevation_gain | 값: 367.00000 | 자료형: str
average_speed | 값: 26.60649 | 자료형: str
average_power | 값: ['188.85205', '12808.00000'] | 자료형: list
max_power | 값: 611.00000 | 자료형: str
coggan_np | 값: ['237.99715', '16138.08000'] | 자료형: list
cp_setting | 값: 200.00000 | 자료형: str
average_hr | 값: ['146.91667', '960.00000'] | 자료형: list
max_heartrate | 값: 184.00000 | 자료형: str
average_cad | 값: ['88.72929', '1919.00000'] | 자료형: list
max_cadence | 값: 98.00000 | 자료형: str
coggan_if | 값: ['0.86544', '16138.08000'] | 자료형: list
coggan_tss | 값: 335.75887 | 자료형: str


In [104]:
for column in selected_metric_columns:
    type_counts = (
        analysis_df[column]
        .dropna()
        .map(type)
        .value_counts()
    )

    print(f"{type_counts}\n")

workout_time
<class 'str'>    591
Name: count, dtype: int64

time_riding
<class 'str'>    574
Name: count, dtype: int64

total_distance
<class 'str'>    563
Name: count, dtype: int64

elevation_gain
<class 'str'>    496
Name: count, dtype: int64

average_speed
<class 'str'>    563
Name: count, dtype: int64

average_power
<class 'list'>    469
Name: count, dtype: int64

max_power
<class 'str'>    469
Name: count, dtype: int64

coggan_np
<class 'list'>    469
Name: count, dtype: int64

cp_setting
<class 'str'>    592
Name: count, dtype: int64

average_hr
<class 'list'>    465
<class 'str'>       3
Name: count, dtype: int64

max_heartrate
<class 'str'>    465
Name: count, dtype: int64

average_cad
<class 'list'>    532
Name: count, dtype: int64

max_cadence
<class 'str'>    532
Name: count, dtype: int64

coggan_if
<class 'list'>    470
Name: count, dtype: int64

coggan_tss
<class 'str'>    490
Name: count, dtype: int64



In [105]:
average_hr_is_string = analysis_df["average_hr"].map(type) == str

analysis_df.loc[
    average_hr_is_string,
    [
        "date",
        "data",
        "average_hr",
        "max_heartrate",
        "workout_time",
    ]
]

,date,data,average_hr,max_heartrate,workout_time
392,2012/01/04 15:35:51 UTC,---------------,148.00000,NaN,4200.00000
393,2012/01/05 12:32:51 UTC,---------------,145.00000,NaN,5820.00000
394,2012/01/06 13:22:53 UTC,---------------,145.00000,NaN,2700.00000


### 자료형 확인 결과

`average_hr`의 유효한 값 대부분은 리스트이지만, 3개는 문자열로
저장되어 있다.

해당 3개 라이드는 `data`에 센서 기록이 표시되어 있지 않고
최대 심박수도 비어 있다. 따라서 시계열 심박 데이터에서 계산된 값이
아니라, 수동으로 입력했거나 외부에서 가져온 요약값일 가능성이 높다.

지금은 삭제하지 않고 이후 데이터 품질을 분류할 때 별도로 구분한다.

## 5. 지표의 자료형 확인과 변환

선택한 지표의 실제 값을 확인한 결과, 숫자가 다음 두 가지 형태로
저장되어 있었다.

- 숫자를 나타내는 문자열: `"4316.00000"`
- 지표값과 관측 정보를 담은 리스트: `["145.75371", "4316.00000"]`

리스트의 첫 번째 요소는 분석에 사용할 지표값이고, 두 번째 요소는
평균 계산에 사용된 관측값 수 또는 가중치 정보로 보인다.

`average_hr`에서는 대부분의 값이 리스트였지만 3개는 문자열이었다.
이 3개 라이드는 센서 정보를 나타내는 `data`가 비어 있고 최대 심박수도
기록되어 있지 않았다. 따라서 센서 시계열에서 계산된 값이 아니라
수동으로 입력했거나 외부에서 가져온 요약값일 가능성이 높다.

원본 구조를 보존하기 위해 `analysis_df`는 변경하지 않는다.
대신 `cleaned_df`를 복사하여 다음과 같이 변환한다.

- 리스트는 첫 번째 요소를 사용한다.
- 숫자 형태의 문자열은 실제 숫자로 변환한다.
- 변환할 수 없는 값은 결측값으로 처리한다.
- 리스트의 두 번째 요소는 필요할 경우 원본 데이터에서 다시 확인한다.

In [106]:
def extract_metric_value(value):
    if isinstance(value, list):
        value = value[0]

    return pd.to_numeric(value, errors="coerce")

In [107]:
cleaned_df = analysis_df.copy()

for column in selected_metric_columns:
    cleaned_df[column] = cleaned_df[column].map(
        extract_metric_value
    )

print(cleaned_df.dtypes)
display(cleaned_df.head())

data                  str
date                  str
sport                 str
workout_time      float64
time_riding       float64
total_distance    float64
elevation_gain    float64
average_speed     float64
average_power     float64
max_power         float64
coggan_np         float64
cp_setting        float64
average_hr        float64
max_heartrate     float64
average_cad       float64
max_cadence       float64
coggan_if         float64
coggan_tss        float64
dtype: object


,data,date,sport,workout_time,time_riding,total_distance,elevation_gain,average_speed,average_power,max_power,coggan_np,cp_setting,average_hr,max_heartrate,average_cad,max_cadence,coggan_if,coggan_tss
0,TDS-H--A-L-----,2005/06/25 14:26:00 UTC,Bike,4800.0,4780.0,35.3275,367.0,26.60649,NaN,NaN,NaN,200.0,146.91667,184.0,NaN,NaN,NaN,NaN
1,TDS-H--A-L-----,2005/06/27 08:39:00 UTC,Bike,6476.0,6325.0,35.0140,467.5,19.92892,NaN,NaN,NaN,200.0,124.97267,182.0,NaN,NaN,NaN,NaN
2,TDS-HC-A-L-----,2005/06/28 17:44:00 UTC,Bike,2156.0,2156.0,17.6530,92.0,29.71052,NaN,NaN,NaN,200.0,157.04592,168.0,88.72929,98.0,NaN,NaN
3,TDS-H--A-L-----,2005/07/06 18:02:00 UTC,Bike,6360.0,6320.0,31.0495,512.0,17.68642,NaN,NaN,NaN,200.0,120.05660,182.0,NaN,NaN,NaN,NaN
4,TDS-H--A-L-----,2005/07/10 09:39:00 UTC,Bike,4680.0,4660.0,32.7700,471.0,25.31588,NaN,NaN,NaN,200.0,146.42735,177.0,NaN,NaN,NaN,NaN


## 6. 선택한 지표의 결측값 확인

숫자로 변환한 `cleaned_df`를 이용하여 각 후보 지표의 결측 개수와
결측 비율을 확인한다.

결측값의 분포를 통해 실제 분석에 사용할 수 있는 지표와
별도의 품질 기준이 필요한 지표를 판단한다.

In [108]:
missing_count = cleaned_df[selected_metric_columns].isna().sum()
missing_ratio = cleaned_df[selected_metric_columns].isna().mean() * 100

missing_summary_df = pd.DataFrame({
    "missing_count": missing_count,
    "missing_ratio": missing_ratio,
})

missing_summary_df.round(2)

,missing_count,missing_ratio
workout_time,1,0.17
time_riding,18,3.04
total_distance,29,4.90
elevation_gain,96,16.22
average_speed,29,4.90
average_power,123,20.78
max_power,123,20.78
coggan_np,123,20.78
cp_setting,0,0.00
average_hr,124,20.95


### 결측값 확인 결과

파워와 심박 관련 지표는 전체 자전거 기록의 약 20%에서 결측값으로
나타났다. 모든 라이딩에 파워 미터와 심박 센서가 사용된 것은 아니기
때문으로 보인다.

IF도 파워 지표와 비슷한 결측 비율을 보였다. 반면 TSS는 파워보다
결측 비율이 낮았다. 일부 TSS는 파워 센서 데이터로부터 자동 계산된
값이 아니라 수동으로 입력했거나 별도의 방식으로 추정한 값일 가능성이
있다.

따라서 결측값이 있는 라이드를 모두 삭제하지 않고, 분석 목적과
센서 유무에 따라 사용할 라이드를 구분할 필요가 있다.

## 7. 센서 표시와 실제 지표 비교

`data`에 표시된 파워 및 심박 센서의 유무와 실제 요약 지표의
존재 여부가 일치하는지 확인한다.

이를 통해 센서 데이터에서 계산된 값과 수동으로 입력된 것으로 보이는
요약값을 구분할 기준을 마련한다.

In [109]:
cleaned_df["has_power_sensor"] = (
    cleaned_df["data"].str.contains("P", na=False)
)

cleaned_df["has_hr_sensor"] = (
    cleaned_df["data"].str.contains("H", na=False)
)

sensor_metric_summary_df = pd.DataFrame(
    {
        "sensor_count": [
            cleaned_df["has_power_sensor"].sum(),
            cleaned_df["has_hr_sensor"].sum(),
        ],
        "metric_count": [
            cleaned_df["average_power"].notna().sum(),
            cleaned_df["average_hr"].notna().sum(),
        ],
    },
    index=["power", "heart_rate"],
)

sensor_metric_summary_df

,sensor_count,metric_count
power,469,469
heart_rate,465,468


In [110]:
cleaned_df["has_power_metric"] = (
    cleaned_df["average_power"].notna()
)

cleaned_df["has_hr_metric"] = (
    cleaned_df["average_hr"].notna()
)

power_mismatch = (
    cleaned_df["has_power_sensor"]
    != cleaned_df["has_power_metric"]
)

hr_mismatch = (
    cleaned_df["has_hr_sensor"]
    != cleaned_df["has_hr_metric"]
)

print("파워 불일치:", power_mismatch.sum())
print("심박 불일치:", hr_mismatch.sum())

파워 불일치: 0
심박 불일치: 3


In [111]:
cleaned_df.loc[
    hr_mismatch,
    [
        "date",
        "data",
        "average_hr",
        "max_heartrate",
        "has_hr_sensor",
        "has_hr_metric",
    ]
]

,date,data,average_hr,max_heartrate,has_hr_sensor,has_hr_metric
392,2012/01/04 15:35:51 UTC,---------------,148.0,NaN,False,True
393,2012/01/05 12:32:51 UTC,---------------,145.0,NaN,False,True
394,2012/01/06 13:22:53 UTC,---------------,145.0,NaN,False,True


### 센서와 지표 비교 결과

파워 센서가 표시된 라이드는 469개이며, 평균 파워가 존재하는
라이드도 469개였다. 행 단위로 비교한 결과 불일치는 없었다.

심박 센서가 표시된 라이드는 465개이지만, 평균 심박이 존재하는
라이드는 468개였다. 행 단위 비교 결과 3개의 불일치가 확인되었다.

이 3개 라이드는 심박 센서 표시 없이 평균 심박만 존재하며, 앞서 확인한
문자열 형태의 평균 심박 기록과 일치한다. 따라서 센서 시계열에서 계산된
값이 아니라 수동 입력 또는 외부에서 가져온 요약값으로 판단한다.

## 8. 날짜 자료형 변환

현재 `date`는 문자열로 저장되어 있다. 시간순 정렬과 기간별 훈련
흐름 분석을 위해 pandas의 날짜 자료형으로 변환한다.

원본 날짜는 UTC 기준이므로 변환 후에도 UTC 시간대를 유지한다.

In [112]:
cleaned_df["date"] = pd.to_datetime(
    cleaned_df["date"],
    format="%Y/%m/%d %H:%M:%S UTC",
    utc=True,
    errors="coerce",
)

print("자료형:", cleaned_df["date"].dtype)
print("변환 실패:", cleaned_df["date"].isna().sum())
print("첫 운동:", cleaned_df["date"].min())
print("마지막 운동:", cleaned_df["date"].max())

자료형: datetime64[us, UTC]
변환 실패: 0
첫 운동: 2005-06-25 14:26:00+00:00
마지막 운동: 2017-07-18 11:01:20+00:00


### 날짜 변환 결과

592개 자전거 기록의 날짜가 모두 정상적으로 변환되었으며,
변환에 실패한 기록은 없었다.

데이터는 2005년 6월 25일부터 2017년 7월 18일까지 약 12년의
운동 기록을 포함한다. 이후 날짜를 기준으로 정렬하고 주간·월간
훈련 흐름을 분석할 수 있다.

### 날짜순 정렬

날짜 변환 후 기록이 시간순으로 정렬되어 있는지 확인한다.
이후 기간별 훈련 흐름을 올바르게 계산할 수 있도록 날짜를 기준으로
오름차순 정렬하고 행 인덱스를 다시 설정한다.

In [113]:
print(
    "현재 날짜순 정렬:",
    cleaned_df["date"].is_monotonic_increasing
)

현재 날짜순 정렬: True


### 날짜순 정렬 확인

날짜가 오름차순으로 정렬되어 있는지 확인한 결과 `True`가
나타났다. 현재 데이터가 이미 과거부터 최근 순서로 정렬되어 있으므로
별도의 정렬은 수행하지 않는다.

## 9. 센서 없는 심박 요약 기록 표시

심박 센서 표시는 없지만 평균 심박 지표가 존재하는 라이드를
별도의 불리언 열로 표시한다.

이 기록은 수동 입력 또는 외부에서 가져온 요약값일 가능성이 있지만,
현재 데이터만으로 출처를 확정할 수 없으므로 관찰된 사실을 나타내는
이름을 사용한다.

In [114]:
cleaned_df["hr_metric_without_sensor"] = (
    ~cleaned_df["has_hr_sensor"]
    & cleaned_df["has_hr_metric"]
)

print(
    "센서 없는 심박 요약:",
    cleaned_df["hr_metric_without_sensor"].sum()
)

cleaned_df.loc[
    cleaned_df["hr_metric_without_sensor"],
    [
        "date",
        "data",
        "average_hr",
        "max_heartrate",
    ]
]

센서 없는 심박 요약: 3


,date,data,average_hr,max_heartrate
392,2012-01-04 15:35:51+00:00,---------------,148.0,NaN
393,2012-01-05 12:32:51+00:00,---------------,145.0,NaN
394,2012-01-06 13:22:53+00:00,---------------,145.0,NaN


### 품질 표시 결과

심박 센서 표시는 없지만 평균 심박이 존재하는 기록은 3개였다.
모두 기존에 확인한 문자열 형태의 평균 심박 기록과 일치하며,
최대 심박은 결측값으로 나타났다.

이 기록을 `hr_metric_without_sensor`로 표시하여 이후 센서 기반
심박 분석에서는 구분할 수 있도록 한다. 다만 수동 입력 여부는
확정하지 않는다.

## 10. 파워 센서 없는 TSS 기록 표시

파워 센서 표시는 없지만 TSS가 존재하는 라이드를 구분한다.
이 값은 파워 시계열에서 직접 계산된 값이 아닐 가능성이 있으므로
`tss_without_power_sensor` 품질 표시를 추가한다.

In [115]:
cleaned_df["has_tss_metric"] = (
    cleaned_df["coggan_tss"].notna()
)

cleaned_df["tss_without_power_sensor"] = (
    ~cleaned_df["has_power_sensor"]
    & cleaned_df["has_tss_metric"]
)

print(
    "파워 센서 없는 TSS:",
    cleaned_df["tss_without_power_sensor"].sum()
)

cleaned_df.loc[
    cleaned_df["tss_without_power_sensor"],
    [
        "date",
        "data",
        "average_power",
        "coggan_np",
        "coggan_if",
        "coggan_tss",
    ],
]

파워 센서 없는 TSS: 21


,date,data,average_power,coggan_np,coggan_if,coggan_tss
392,2012-01-04 15:35:51+00:00,---------------,NaN,NaN,NaN,40.0
393,2012-01-05 12:32:51+00:00,---------------,NaN,NaN,NaN,80.0
394,2012-01-06 13:22:53+00:00,---------------,NaN,NaN,NaN,40.0
400,2012-04-01 14:30:30+00:00,TDS--C-AGL-----,NaN,NaN,NaN,100.0
420,2012-07-15 15:56:23+00:00,TDS--C-AGL-----,NaN,NaN,NaN,130.0
421,2012-07-20 10:54:12+00:00,TDS--C-AGL-----,NaN,NaN,NaN,75.0
422,2012-07-22 11:31:12+00:00,TDS--C-AGL-----,NaN,NaN,NaN,115.0
427,2012-07-28 09:24:16+00:00,TDS--C-AGL-----,NaN,NaN,0.94,88.0
431,2012-08-04 11:55:47+00:00,TDS--C-AGL-----,NaN,NaN,NaN,130.0
432,2012-08-05 13:27:44+00:00,TDS--C-AGL-----,NaN,NaN,NaN,75.0


### TSS 품질 표시 결과

파워 센서 없이 TSS가 존재하는 기록은 21개였다. 이 기록들은 모두
평균 파워와 NP가 비어 있었으며, IF는 1개 기록에만 존재했다.

TSS는 대부분 둥근 값으로 저장되어 있어 수동 입력, 외부에서 가져온
요약값 또는 추정값일 가능성이 있다. 장기 훈련 부하 분석에는 참고할
수 있지만, 파워 기반 분석에서는 별도로 구분한다.

## 11. 주요 지표의 범위 확인

주요 운동 지표의 최솟값, 중앙값, 최댓값 등을 확인하여 데이터의
전체적인 분포를 파악하고 이상치 후보를 찾는다.

이 단계에서는 값을 삭제하거나 수정하지 않고, 추가 확인이 필요한
기록을 찾는 데 집중한다.

In [116]:
range_check_columns = [
    "workout_time",
    "total_distance",
    "average_power",
    "max_power",
    "average_hr",
    "max_heartrate",
    "coggan_if",
    "coggan_tss",
]

range_summary_df = (
    cleaned_df[range_check_columns]
    .describe()
    .T
    .round(2)
)

range_summary_df

,count,mean,std,min,25%,50%,75%,max
workout_time,591.0,6225.83,4160.61,126.00,4260.00,5402.00,6757.93,40169.00
total_distance,563.0,42.87,26.36,0.00,29.37,39.70,49.81,197.18
average_power,469.0,171.85,36.10,37.48,147.91,169.55,196.51,274.39
max_power,469.0,588.92,117.11,129.00,515.00,591.00,661.00,885.00
average_hr,468.0,136.24,16.37,14.00,129.00,138.22,146.32,175.40
max_heartrate,465.0,167.93,17.97,14.00,162.00,171.00,177.00,229.00
coggan_if,470.0,0.84,0.12,0.24,0.78,0.86,0.92,1.13
coggan_tss,490.0,118.03,69.09,1.42,76.51,110.53,138.96,674.50


### 지표 범위 확인 결과

평균 심박과 최대 심박의 최솟값이 모두 14 bpm으로 나타났다.
라이딩 중 정상적인 심박수로 보기 어려우므로 센서 오류 또는 기록
문제일 가능성이 높다.

최대 심박의 최댓값인 229 bpm도 센서 오류 가능성이 있어 해당
라이드를 별도로 확인할 필요가 있다.

IF는 0.24부터 1.13, TSS는 1.42부터 674.50까지 넓은 범위를
보였다. 이러한 값은 운동 시간과 강도에 따라 실제로 나타날 수도
있으므로 값 하나만으로 이상치라고 판단하지 않는다.

다음 분석에서는 의심스러운 심박 기록을 직접 확인하고, IF와 TSS를
운동 시간 및 파워 지표와 함께 비교한다. 현재 단계에서는 어떤 값도
삭제하거나 수정하지 않는다.

## 12. 심박 이상치 후보 확인

요약 통계에서 발견한 비정상적으로 낮거나 높은 심박 기록을
직접 확인한다.

넓은 범위의 검토 조건을 사용하여 후보를 찾으며, 현재 단계에서는
해당 기록을 삭제하거나 실제 오류로 확정하지 않는다.

In [117]:
hr_range_needs_review = (
    (cleaned_df["average_hr"] < 40)
    | (cleaned_df["max_heartrate"] < 40)
    | (cleaned_df["max_heartrate"] > 220)
    | (cleaned_df["average_hr"] > cleaned_df["max_heartrate"])
)

print(
    "심박 검토 대상:",
    hr_range_needs_review.sum()
)

cleaned_df.loc[
    hr_range_needs_review,
    [
        "date",
        "data",
        "workout_time",
        "average_hr",
        "max_heartrate",
    ]
]

심박 검토 대상: 5


,date,data,workout_time,average_hr,max_heartrate
12,2005-08-28 23:06:00+00:00,TDS-H--A-L-----,1800.0,14.00000,14.0
14,2005-09-11 22:40:00+00:00,TDS-H--A-L-E---,6120.0,14.00000,14.0
42,2006-10-28 14:13:00+00:00,TDS-H--A-L-----,6360.0,26.00000,70.0
102,2007-06-04 17:56:03+00:00,T---H--A-------,5065.0,136.38401,228.0
317,2009-07-16 16:01:07+00:00,TDSPHC-AGL-----,6246.0,141.29711,229.0


In [118]:
cleaned_df["hr_range_needs_review"] = hr_range_needs_review

### 심박 이상치 후보 확인 결과

심박 범위 검토 대상으로 5개 라이드가 확인되었다. 모든 기록에는
심박 센서 표시가 존재했다.

- 평균·최대 심박이 모두 14 bpm인 기록: 2개
- 평균 심박이 26 bpm인 기록: 1개
- 최대 심박이 228 bpm 또는 229 bpm인 기록: 2개

14 bpm과 26 bpm은 라이딩 중 정상적인 평균 심박으로 보기 어려워
센서 또는 기록 오류일 가능성이 높다.

최대 심박이 높은 두 라이드는 평균 심박은 정상적인 범위였다.
원본 시계열을 확인한 결과 운동 초반에 높은 심박이 일정 시간 이어져,
짧은 구간의 센서 측정 오류일 가능성이 있다.

해당 라이드를 삭제하지 않고 `hr_range_needs_review` 열에 표시한다.
이후 심박 분석에서는 평균 심박과 최대 심박의 품질을 구분해서
판단할 필요가 있다.

## 13. IF와 TSS 계산 검증

`coggan_if` 리스트의 두 번째 요소를 별도로 추출하여
TSS 계산에 사용된 시간 또는 가중치 역할을 하는지 확인한다.

In [142]:
def extract_metric_weight(value):
    if isinstance(value, list) and len(value) > 1:
        return value[1]

    return None


cleaned_df["coggan_if_weight"] = pd.to_numeric(
    analysis_df["coggan_if"].map(
        extract_metric_weight
    ),
    errors="coerce",
)

cleaned_df[
   [
        "workout_time",
        "time_riding",
        "coggan_if",
        "coggan_if_weight",
        "coggan_tss",
    ]
].dropna().head()

,workout_time,time_riding,coggan_if,coggan_if_weight,coggan_tss
58,16138.16,16075.08,0.86544,16138.08,335.75887
76,15027.26,14939.82,0.91280,15026.76,347.78921
77,23940.26,23790.06,0.82586,23940.00,453.56478
79,3549.00,2816.00,1.01068,2817.00,79.93059
81,3549.42,3548.16,1.02252,3549.42,103.08512


In [148]:
# 계산 TSS = coggan_if_weight ÷ 3600 × coggan_if² × 100

cleaned_df["tss_from_formula"] = (
    cleaned_df["coggan_if_weight"]
    / 3600
    * cleaned_df["coggan_if"] ** 2
    * 100
)

cleaned_df["tss_absolute_difference"] = (
    cleaned_df["coggan_tss"]
    - cleaned_df["tss_from_formula"]
).abs()

In [150]:
power_tss_validation_mask = (
    cleaned_df["has_power_sensor"]
    & cleaned_df["coggan_tss"].notna()
    & cleaned_df["tss_from_formula"].notna()
)

cleaned_df.loc[
    power_tss_validation_mask,
    "tss_absolute_difference",
].describe().round(4)

count    469.0000
mean       0.0007
std        0.0007
min        0.0000
25%        0.0003
50%        0.0006
75%        0.0010
max        0.0058
Name: tss_absolute_difference, dtype: float64

### IF와 TSS 계산 검증 결과

`coggan_if` 리스트의 두 번째 요소는 라이드에 따라 `workout_time` 또는
`time_riding`에 가까운 값을 보였다. 따라서 특정 시간 열을 그대로 복사한
값이라기보다 IF와 TSS 계산에 실제로 사용된 시간 가중치로 해석하고
`coggan_if_weight` 열에 보존한다.

`coggan_if_weight / 3600 * coggan_if² * 100`으로 TSS를 다시 계산한 뒤,
파워 센서가 있는 469개 라이드의 저장된 TSS와 비교했다.

- 평균 절대 오차: 0.0007
- 중앙 절대 오차: 0.0006
- 최대 절대 오차: 0.0058

오차가 매우 작아 소수점 저장 과정의 반올림 차이로 볼 수 있다. 따라서
파워 센서가 있는 라이드의 TSS는 IF와 계산 가중 시간을 이용한 공식과
일관되게 계산된 것으로 판단한다.

파워 센서 없이 TSS가 존재하는 21개 기록은 계산에 필요한 값이 부족하므로
이번 공식 검증 대상에서 제외하고 별도의 품질 표시를 유지한다.

In [164]:
cleaned_df["if_from_formula"] = (
    cleaned_df["coggan_np"]
    / cleaned_df["cp_setting"]
)

cleaned_df["if_absolute_difference"] = (
    cleaned_df["coggan_if"]
    - cleaned_df["if_from_formula"]
).abs()

cleaned_df[
    "if_absolute_difference"
].dropna().describe().round(6)

count    469.000000
mean       0.012576
std        0.026495
min        0.000000
25%        0.000002
50%        0.000003
75%        0.000005
max        0.089970
Name: if_absolute_difference, dtype: float64

In [165]:
if_difference_needs_review = (
    cleaned_df["if_absolute_difference"] > 0.001
)

print(
    "IF 기준 불일치:",
    if_difference_needs_review.sum()
)

IF 기준 불일치: 90


In [171]:
cleaned_df["if_reference_from_stored"] = (
    cleaned_df["coggan_np"]
    / cleaned_df["coggan_if"]
)

cleaned_df.loc[
    if_difference_needs_review,
    [
        "date",
        "coggan_np",
        "cp_setting",
        "if_reference_from_stored",
        "coggan_if",
        "if_from_formula",
        "if_absolute_difference",
    ]
].nlargest(
    10,
    "if_absolute_difference"
)

,date,coggan_np,cp_setting,if_reference_from_stored,coggan_if,if_from_formula,if_absolute_difference
566,2015-08-09 09:59:09+00:00,202.42348,225.0,250.001210,0.80969,0.899660,0.089970
567,2015-08-11 11:08:50+00:00,201.04297,225.0,250.000584,0.80417,0.893524,0.089354
570,2015-08-28 14:30:59+00:00,192.92059,225.0,250.000765,0.77168,0.857425,0.085745
569,2015-08-16 12:38:21+00:00,191.20816,225.0,250.000863,0.76483,0.849814,0.084984
571,2015-09-06 11:50:38+00:00,189.15605,225.0,250.001388,0.75662,0.840694,0.084074
380,2010-10-13 17:40:31+00:00,241.48951,255.0,235.001129,1.02761,0.947018,0.080592
580,2016-08-10 18:21:42+00:00,179.04927,225.0,249.998981,0.71620,0.795775,0.079575
384,2010-10-20 16:55:43+00:00,238.21560,255.0,235.000789,1.01368,0.934179,0.079501
369,2010-09-22 16:11:33+00:00,234.37438,255.0,234.999479,0.99734,0.919115,0.078225
575,2016-05-05 11:57:18+00:00,173.68722,225.0,249.999597,0.69475,0.771943,0.077193


In [172]:
cleaned_df["if_reference_rounded"] = (
    cleaned_df["if_reference_from_stored"].round()
)

if_reference_patterns = (
    cleaned_df.loc[
        if_difference_needs_review,
        [
            "cp_setting",
            "if_reference_rounded",
        ]
    ]
    .value_counts()
)

if_reference_patterns

cp_setting  if_reference_rounded
255.0       235.0                   64
225.0       250.0                   22
280.0       275.0                    2
281.0       275.0                    1
239.0       235.0                    1
Name: count, dtype: int64

In [181]:
if_reference_periods = (
    cleaned_df.loc[
        if_difference_needs_review,
        [
            "date",
            "cp_setting",
            "if_reference_rounded",
        ],
    ]
    .groupby(
        [
            "cp_setting",
            "if_reference_rounded",
        ]
    )["date"]
    .agg(["count", "min", "max"])
)

if_reference_periods

,,count,min,max
cp_setting,if_reference_rounded,,,
225.0,250.0,22,2015-08-07 13:37:54+00:00,2016-08-29 10:14:46+00:00
239.0,235.0,1,2010-10-11 11:35:49+00:00,2010-10-11 11:35:49+00:00
255.0,235.0,64,2010-09-01 17:15:34+00:00,2012-07-30 17:48:22+00:00
280.0,275.0,2,2007-05-13 05:20:15+00:00,2007-07-03 12:18:18+00:00
281.0,275.0,1,2009-05-13 18:18:39+00:00,2009-05-13 18:18:39+00:00


In [186]:
reference_period_mask = (
    cleaned_df["date"].between(
        "2010-09-01",
        "2012-07-30 23:59:59"
    )
    & cleaned_df["has_power_sensor"]
)

print("기간 내 파워 라이드:")
print(reference_period_mask.sum())

print("\n역산 기준 파워:")
print(
    cleaned_df.loc[
        reference_period_mask,
        "if_reference_rounded",
    ]
    .value_counts()
)

print("\n저장된 cp_setting:")
print(
    cleaned_df.loc[
        reference_period_mask,
        "cp_setting",
    ]
    .value_counts()
)

기간 내 파워 라이드:
65

역산 기준 파워:
if_reference_rounded
235.0    65
Name: count, dtype: int64

저장된 cp_setting:
cp_setting
255.0    64
239.0     1
Name: count, dtype: int64


### IF 계산 검증 결과

`coggan_np / cp_setting`으로 IF를 다시 계산하여 저장된 `coggan_if`와
비교했다. NP가 존재하는 469개 라이드 중 대부분은 두 값이 소수점
반올림 수준에서 일치했지만, 절대 오차가 0.001보다 큰 기록이 90개
확인되었다. 최대 절대 오차는 약 0.09로 반올림만으로 설명할 수 없다.

저장된 IF에서 `coggan_np / coggan_if`로 기준 파워를 역산한 결과,
불일치 기록은 몇 가지 반복 패턴으로 묶였다.

- `cp_setting` 255W, 역산 기준 235W: 64개
- `cp_setting` 225W, 역산 기준 250W: 22개
- `cp_setting` 280W, 역산 기준 275W: 2개
- 그 밖의 조합: 2개

특히 2010년 9월 1일부터 2012년 7월 30일까지 파워 센서가 있는
65개 라이드 모두 IF 역산 기준이 235W였다. 이 중 64개는
`cp_setting`이 255W이고 1개는 239W였다. 같은 기준이 특정 기간에
연속적으로 사용되었으므로 개별 라이드의 우연한 계산 오류보다는
별도의 FTP 또는 Coggan 계산 기준이 설정되어 있었을 가능성이 높다.

따라서 `cp_setting`이 항상 IF 계산의 분모라고 가정하지 않는다. 원본
`cp_setting`과 역산한 기준 파워를 모두 보존하며, 역산값은 저장된 IF를
설명하기 위한 파생값으로만 사용한다. 역산값은 NP와 IF에서 만들어진
값이므로 실제 체력 기준의 독립적인 정답이나 머신러닝 입력값으로
그대로 사용하지 않는다.

In [191]:
cleaned_df["workout_hours"] = (
        cleaned_df["workout_time"]
        / 3600
)

if_context_columns = [
    "date",
    "workout_hours",
    "average_power",
    "coggan_np",
    "cp_setting",
    "if_reference_rounded",
    "coggan_if",
    "coggan_tss",
]

print("IF가 높은 라이드")

display(
    cleaned_df.nlargest(
        5,
        "coggan_if",
    )[if_context_columns]
)

print("IF가 낮은 라이드")

display(
    cleaned_df.nsmallest(
        5,
        "coggan_if",
    )[if_context_columns]
)

IF가 높은 라이드


,date,workout_hours,average_power,max_power,coggan_np,cp_setting,if_reference_rounded,coggan_if,coggan_tss
130,2008-07-22 16:44:44+00:00,0.330278,250.70227,879.0,282.12618,250.0,250.0,1.12850,42.06163
128,2008-07-16 17:57:26+00:00,0.325556,248.56143,836.0,281.26070,250.0,250.0,1.12504,41.20626
352,2010-08-27 15:41:41+00:00,1.361389,200.04754,651.0,223.36018,200.0,200.0,1.11680,169.79845
517,2014-11-21 17:19:42+00:00,0.163056,217.00511,433.0,239.78078,220.0,220.0,1.08991,19.36953
180,2008-12-21 14:49:35+00:00,1.296944,242.25294,627.0,264.14502,250.0,250.0,1.05658,144.78588


IF가 낮은 라이드


,date,workout_hours,average_power,max_power,coggan_np,cp_setting,if_reference_rounded,coggan_if,coggan_tss
228,2009-03-15 11:00:04+00:00,0.390278,37.47568,497.0,67.00193,275.0,275.0,0.24364,2.30523
126,2008-05-18 21:38:31+00:00,0.086398,101.00965,129.0,101.25792,250.0,250.0,0.40503,1.41722
245,2009-04-06 09:14:47+00:00,0.848333,102.35828,430.0,114.89606,275.0,275.0,0.41780,14.64365
578,2016-08-07 10:34:21+00:00,0.909722,55.48748,428.0,115.54874,225.0,250.0,0.46219,19.43387
115,2007-11-19 18:43:45+00:00,1.001556,146.08800,220.0,147.81254,275.0,275.0,0.53750,29.55012


### IF 극단값 확인 결과

IF가 높은 라이드와 낮은 라이드를 운동 시간, 파워, 기준 파워 및
TSS와 함께 확인했다.

높은 IF 상위 기록은 대부분 비교적 짧은 고강도 라이드였으며, 저장된
`cp_setting`과 IF에서 역산한 기준 파워도 일치했다. 따라서 기준 파워
불일치 때문에 IF가 비정상적으로 높아진 것으로 보이지 않는다. 다만
약 1시간 이상 지속된 높은 IF 기록은 당시 기준 파워가 실제 체력보다
낮게 설정되었거나 경기·테스트였을 가능성을 추가로 고려할 수 있다.

낮은 IF 상위 기록도 짧거나 평균 파워와 NP가 낮은 라이드였다. 578번
라이드는 `cp_setting` 225W와 역산 기준 250W가 달랐지만, 225W로 다시
계산해도 IF는 약 0.51로 낮다. 따라서 기준 파워 차이가 저장된 IF를 더
낮추기는 했지만 낮은 강도의 주된 원인은 아니다.

최저 IF 라이드는 최대 파워가 497W였지만 평균 파워 37W, NP 67W로
대부분의 시간에는 매우 가볍게 이동한 것으로 보인다. 최대 파워가
잠시 높더라도 운동 전체의 NP가 낮으면 IF는 낮게 계산될 수 있다.

현재 IF 범위 0.24~1.13은 운동 시간과 전체 강도로 설명할 수 있으므로
범위만을 이유로 값을 삭제하거나 수정하지 않는다. 다만 파워 센서와
NP 없이 IF만 존재하는 기록 1개는 계산을 검증할 수 없으므로 기존의
품질 구분을 유지한다.

In [193]:
tss_context_columns = [
    "date",
    "workout_hours",
    "average_power",
    "coggan_np",
    "coggan_if",
    "coggan_tss",
    "has_power_sensor",
    "tss_without_power_sensor",
]

print("TSS가 높은 라이드")

display(
    cleaned_df.nlargest(
        5,
        "coggan_tss",
    )[tss_context_columns]
)

print("TSS가 낮은 라이드")

display(
    cleaned_df.nsmallest(
        5,
        "coggan_tss",
    )[tss_context_columns]
)

TSS가 높은 라이드


,date,workout_hours,average_power,coggan_np,coggan_if,coggan_tss,has_power_sensor,tss_without_power_sensor
96,2007-05-13 05:20:15+00:00,8.409722,190.12172,246.28281,0.89557,674.50390,True,False
97,2007-05-19 23:00:00+00:00,7.992794,172.10396,224.28974,0.81560,509.27185,True,False
93,2007-04-21 23:00:00+00:00,6.452711,193.11846,235.79961,0.85745,474.41185,True,False
313,2009-07-04 05:09:48+00:00,11.158056,132.64348,176.41322,0.64150,459.14827,True,False
77,2007-04-07 23:00:00+00:00,6.650072,179.75574,227.11276,0.82586,453.56478,True,False


TSS가 낮은 라이드


,date,workout_hours,average_power,coggan_np,coggan_if,coggan_tss,has_power_sensor,tss_without_power_sensor
126,2008-05-18 21:38:31+00:00,0.086398,101.00965,101.25792,0.40503,1.41722,True,False
562,2015-07-21 11:19:33+00:00,0.035000,126.90476,145.68628,0.66221,1.53483,True,False
228,2009-03-15 11:00:04+00:00,0.390278,37.47568,67.00193,0.24364,2.30523,True,False
214,2009-02-21 09:47:33+00:00,0.225833,127.31527,158.25539,0.57547,7.46973,True,False
194,2009-01-16 15:37:26+00:00,0.250278,181.26699,196.31860,0.74082,12.56192,True,False


In [194]:
cleaned_df["if_reference_differs_from_cp"] = (
    if_difference_needs_review
)

cleaned_df["tss_using_cp_setting"] = (
    cleaned_df["coggan_if_weight"]
    / 3600
    * cleaned_df["if_from_formula"] ** 2
    * 100
)

cleaned_df["tss_vs_cp_percent_difference"] = (
    (
        cleaned_df["coggan_tss"]
        - cleaned_df["tss_using_cp_setting"]
    )
    / cleaned_df["tss_using_cp_setting"]
    * 100
)

cleaned_df.loc[
    cleaned_df["if_reference_differs_from_cp"],
    [
        "date",
        "cp_setting",
        "if_reference_rounded",
        "coggan_tss",
        "tss_using_cp_setting",
        "tss_vs_cp_percent_difference",
    ],
].head()

,date,cp_setting,if_reference_rounded,coggan_tss,tss_using_cp_setting,tss_vs_cp_percent_difference
96,2007-05-13 05:20:15+00:00,280.0,275.0,674.50390,650.629557,3.669422
107,2007-07-03 12:18:18+00:00,280.0,275.0,90.85627,87.640381,3.669415
273,2009-05-13 18:18:39+00:00,281.0,275.0,130.17876,124.678871,4.411244
356,2010-09-01 17:15:34+00:00,255.0,235.0,109.17243,92.718920,17.745580
357,2010-09-03 16:11:30+00:00,255.0,235.0,115.64436,98.215454,17.745584


### TSS 극단값 확인 결과

TSS가 높은 라이드와 낮은 라이드를 운동 시간과 IF를 함께 비교했다.
TSS 상위 5개 기록은 모두 장시간 라이드였으며, IF도 약 0.64~0.90으로
나타났다. 313번 라이드는 운동 시간이 약 11.2시간으로 가장 길지만
IF가 약 0.64로 비교적 낮아 TSS는 네 번째로 높았다. 이는 TSS가 운동
시간뿐 아니라 IF의 제곱에도 영향을 받는다는 계산 구조와 일치한다.

TSS 하위 기록은 운동 시간이 매우 짧거나 IF가 매우 낮았다. 562번
라이드는 IF가 약 0.66이지만 운동 시간이 약 2.1분에 불과하여 낮은
TSS가 자연스럽다. 반대로 228번 라이드는 약 23.4분 동안 기록되었지만
IF가 약 0.24로 매우 낮아 TSS가 작게 계산되었다.

상위 및 하위 5개 기록에는 모두 파워 센서가 포함되어 있었고, 운동
시간과 IF를 함께 고려했을 때 극단적인 TSS도 설명 가능했다. 따라서
현재 확인한 TSS 극단값은 범위만을 이유로 삭제하거나 수정하지 않는다.
다만 TSS는 기준 파워 설정의 영향을 받으므로, IF 계산 기준이
`cp_setting`과 다른 기록은 기존 품질 표시를 유지한다.